# PerformanceEvaluator – Usage Examples

This notebook shows how to use the `PerformanceEvaluator` class to compare model performance between a reference dataset and a current dataset.  
It supports:

- **Binary classification**: AUC, KS, Accuracy, Precision, Recall, F1  
- **Multiclass classification**: Accuracy, macro/weighted Precision/Recall/F1  
- **Regression**: RMSE, MAE, R2, SMAPE, P90/P95 error, Huber loss  

All metrics are computed for both datasets, and the delta (current – reference) is returned.

In [1]:
import numpy as np
import pandas as pd
import warnings

# The class we want to demonstrate
from momo_ml.monitor.performance import PerformanceEvaluator

# For demonstration we will create synthetic data
np.random.seed(42)

In [2]:
def create_sample_data(task='binary', n=1000, label_col='y', pred_col='y_pred'):
    """Create a simple DataFrame with labels and predictions."""
    if task == 'binary':
        y = np.random.randint(0, 2, size=n)
        y_pred_prob = np.clip(y + np.random.normal(0, 0.3, size=n), 0, 1)
        df = pd.DataFrame({label_col: y, pred_col: y_pred_prob})
    elif task == 'multiclass':
        y = np.random.choice([0,1,2], size=n)
        # predictions are class labels (already hard-classified)
        noise = np.random.choice([-1,0,1], size=n, p=[0.1,0.8,0.1])
        y_pred = np.clip(y + noise, 0, 2)
        df = pd.DataFrame({label_col: y, pred_col: y_pred})
    elif task == 'regression':
        y = np.random.randn(n) * 2 + 5
        y_pred = y + np.random.randn(n) * 0.5
        df = pd.DataFrame({label_col: y, pred_col: y_pred})
    return df

## 1. Binary classification – basic usage

In [4]:
# Create reference and current datasets
ref = create_sample_data('binary', n=1000)
cur = create_sample_data('binary', n=800)   # current set may have different size

# Initialize the evaluator (auto-detects task type)
evaluator = PerformanceEvaluator(
    ref_df=ref,
    cur_df=cur,
    label_col='y',
    pred_col='y_pred'
)

# Run evaluation
result = evaluator.evaluate()

# Show the result
print("Task type:", result['task_type'])
print("Classification subtype:", result['classification_subtype'])
print("\nReference metrics:")
print(result['reference'])
print("\nCurrent metrics:")
print(result['current'])
print("\nDelta (current - reference):")
print(result['delta'])

Task type: classification
Classification subtype: binary

Reference metrics:
{'auc': 0.9902433755760369, 'ks': 0.8942972350230415, 'accuracy': 0.941, 'precision': 0.9396378269617707, 'recall': 0.9415322580645161, 'f1': 0.9405840886203424}

Current metrics:
{'auc': 0.9916455823041661, 'ks': 0.9262005561261555, 'accuracy': 0.96, 'precision': 0.9639423076923077, 'recall': 0.9593301435406698, 'f1': 0.9616306954436451}

Delta (current - reference):
{'ks': 0.031903321103114, 'auc': 0.001402206728129185, 'recall': 0.017797885476153685, 'accuracy': 0.019000000000000017, 'f1': 0.021046606823302727, 'precision': 0.024304480730537037}


## 2. Binary classification with non‑standard labels

In [5]:
# Create data with labels 5 (positive) and 2 (negative)
ref2 = ref.copy()
ref2['y'] = np.where(ref2['y'] == 1, 5, 2)   # 1→5, 0→2
cur2 = cur.copy()
cur2['y'] = np.where(cur2['y'] == 1, 5, 2)

evaluator2 = PerformanceEvaluator(ref2, cur2, 'y', 'y_pred')
result2 = evaluator2.evaluate()

# The evaluator will issue a warning and treat the larger label (5) as positive
print("Reference metrics (non‑standard labels):")
print(result2['reference'])
# Note: precision/recall/F1 are still computed correctly because the library
# internally maps positive to max(label) and negative to min(label).

Reference metrics (non‑standard labels):
{'auc': 0.9902433755760369, 'ks': 0.8942972350230415, 'accuracy': 0.941, 'precision': 0.9396378269617707, 'recall': 0.9415322580645161, 'f1': 0.9405840886203424}


E:\git repo\momo_ml\momo_ml\monitor\performance.py:155: UserWarning: Binary classification detected but labels are [2 5]. For accurate precision/recall/F1 calculation, it is recommended to convert labels to 0 (negative) and 1 (positive). The library will treat 5 as positive and 2 as negative.
  warnings.warn(
E:\git repo\momo_ml\momo_ml\monitor\performance.py:155: UserWarning: Binary classification detected but labels are [2 5]. For accurate precision/recall/F1 calculation, it is recommended to convert labels to 0 (negative) and 1 (positive). The library will treat 5 as positive and 2 as negative.
  warnings.warn(


## 3. Multiclass classification

In [5]:
ref_mc = create_sample_data('multiclass', n=1200)
cur_mc = create_sample_data('multiclass', n=900)

evaluator_mc = PerformanceEvaluator(
    ref_mc, cur_mc, label_col='y', pred_col='y_pred'
)
result_mc = evaluator_mc.evaluate()

print("Task type:", result_mc['task_type'])
print("Classification subtype:", result_mc['classification_subtype'])
print("\nReference metrics (multiclass):")
print(result_mc['reference'])
print("\nDelta:")
print(result_mc['delta'])

Task type: classification
Classification subtype: multiclass

Reference metrics (multiclass):
{'accuracy': 0.8633333333333333, 'precision_macro': 0.86150219062491, 'recall_macro': 0.8625107900627161, 'f1_macro': 0.8617869710777392, 'precision_weighted': 0.8620744286196316, 'recall_weighted': 0.8633333333333333, 'f1_weighted': 0.8624871295806057}

Delta:
{'accuracy': 0.01333333333333342, 'f1_weighted': 0.014114464973167706, 'precision_macro': 0.015055194132322258, 'precision_weighted': 0.014485368172176383, 'recall_weighted': 0.01333333333333342, 'f1_macro': 0.01480383221997128, 'recall_macro': 0.014136678859778096}


## 4. Regression

In [6]:
ref_reg = create_sample_data('regression', n=1500)
cur_reg = create_sample_data('regression', n=1100)

evaluator_reg = PerformanceEvaluator(
    ref_reg, cur_reg, label_col='y', pred_col='y_pred'
)
result_reg = evaluator_reg.evaluate()

print("Task type:", result_reg['task_type'])
print("\nReference metrics (regression):")
print(result_reg['reference'])
print("\nCurrent metrics:")
print(result_reg['current'])
print("\nDelta:")
print(result_reg['delta'])

Task type: regression

Reference metrics (regression):
{'rmse': 0.5263175617912724, 'mae': 0.4179424875503624, 'r2': 0.9327168590400757, 'smape': np.float64(12.788587938153473), 'p90_error': 0.8640998166646531, 'p95_error': 1.024373284548696, 'huber_loss': 0.13577718528222193}

Current metrics:
{'rmse': 0.4797674732060488, 'mae': 0.38327334113839584, 'r2': 0.9399816578358937, 'smape': np.float64(10.330391748716398), 'p90_error': 0.7838017041328017, 'p95_error': 0.9349535422240994, 'huber_loss': 0.11397383675846255}

Delta:
{'huber_loss': -0.021803348523759383, 'p90_error': -0.08029811253185137, 'rmse': -0.04655008858522364, 'r2': 0.007264798795817984, 'p95_error': -0.08941974232459649, 'mae': -0.034669146411966556, 'smape': np.float64(-2.458196189437075)}


## 5. Customising SMAPE and Huber loss

In [7]:
# SMAPE as ratio (0–2) instead of percentage, with a custom epsilon
evaluator_custom = PerformanceEvaluator(
    ref_reg, cur_reg,
    label_col='y',
    pred_col='y_pred',
    smape_as_percentage=False,   # returns ratio
    smape_epsilon=1e-6,
    huber_delta=2.0               # larger delta makes loss more quadratic
)
result_custom = evaluator_custom.evaluate()

print("SMAPE (ratio):", result_custom['current']['smape'])
print("Huber loss (delta=2.0):", result_custom['current']['huber_loss'])

SMAPE (ratio): 0.10330387029000555
Huber loss (delta=2.0): 0.11508841417325837


## 6. Handling missing columns or empty data

In [8]:
# Missing column
ref_bad = ref.copy()
ref_bad = ref_bad.drop(columns=['y_pred'])   # remove prediction column

evaluator_bad = PerformanceEvaluator(ref_bad, cur, 'y', 'y_pred')
result_bad = evaluator_bad.evaluate()
print("Error when column missing:", result_bad.get('error'))

# All NaN values
ref_nan = ref.copy()
ref_nan['y_pred'] = np.nan   # all predictions are NaN
evaluator_nan = PerformanceEvaluator(ref_nan, cur, 'y', 'y_pred')
result_nan = evaluator_nan.evaluate()
print("Error when all NaN:", result_nan.get('error'))

Error when column missing: Missing required columns: ref_df.y_pred
Error when all NaN: No valid (non-NaN) values in label/pred columns.


## 7. Forcing the task type (bypass auto‑detection)

In [9]:
# Even if data looks like classification (few unique values), we can force regression
ref_force = create_sample_data('binary', n=500)
cur_force = create_sample_data('binary', n=500)

evaluator_force = PerformanceEvaluator(
    ref_force, cur_force,
    label_col='y',
    pred_col='y_pred',
    task_type='regression'   # force regression metrics
)
result_force = evaluator_force.evaluate()
print("Forced task type:", result_force['task_type'])
print("Metrics (now regression-like):", list(result_force['reference'].keys()))

Forced task type: regression
Metrics (now regression-like): ['rmse', 'mae', 'r2', 'smape', 'p90_error', 'p95_error', 'huber_loss']


## 8. What if the label or prediction columns are not provided?

In [10]:
evaluator_no_label = PerformanceEvaluator(ref, cur, label_col=None, pred_col='y_pred')
result_no_label = evaluator_no_label.evaluate()
print("Error when label_col=None:", result_no_label.get('error'))

Error when label_col=None: label_col and pred_col must be provided.


## Summary
The `PerformanceEvaluator` provides a unified interface to compare model performance between two datasets.  
It automatically detects the task type (classification vs regression) and returns a dictionary with:
- `task_type`
- `classification_subtype` (for classification tasks)
- `reference` metrics
- `current` metrics
- `delta` (current – reference)

All metrics are computed on non‑NaN values only. The class also gives meaningful warnings or error messages when the data is inconsistent.